# 04 — Koalisyon Tahmin Modeli

**Amaç:** Pretrained GNN encoder üzerine koalisyon skorlama başlığı ekleyip, tarihsel koalisyonları geçerli/geçersiz olarak ayırt etmek.

**Mimari:**
- **Encoder:** 2-katmanlı RGCN (04a'dan pretrained ağırlıklar)
- **Coalition Head:** DeepSets (mean pool) + pairwise bilinear etkileşim terimi → MLP → skor

**Eğitim hedefleri (multi-task):**
1. **Ana:** Contrastive BCE — geçerli vs geçersiz koalisyon
2. **Yardımcı:** Duration regression (pozitif örneklerde) — koalisyon ömrü
3. **Yardımcı:** Cohesion regression (pozitif örneklerde) — oylama uyumu

**Fine-tuning stratejisi:** Aşamalı
- Faz 1 (epoch 1-50): Encoder frozen, sadece coalition head eğitilir
- Faz 2 (epoch 51+): End-to-end, encoder LR = head LR / 10

**Oyun-teorik çıktılar:**
- Shapley değerleri (Monte Carlo yaklaşımı)
- Koalisyon stabilitesi analizi

**Süre:** ~15-20 dakika (Colab T4 GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

START_YEAR = 1946
TRAIN_END = 1999
VAL_END = 2009
END_YEAR = 2016

In [ ]:
!pip install -q torch torch_geometric numpy pandas tqdm scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Veri Yükleme

In [ ]:
# Koalisyon örneklerini yükle
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

print(f'Pozitif örnekler: {len(pos_df)}')
print(f'Negatif örnekler: {len(neg_df)}')
print(f'\nPozitif sütunlar: {list(pos_df.columns)}')
print(f'Negatif sütunlar: {list(neg_df.columns)}')
print(f'\nPozitif örnek (ilk 3):')
print(pos_df.head(3))

In [ ]:
# Snapshot'ları yükle (04a ile aynı format)
EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

def load_snapshot(year):
    """Bir yılın snapshot'ını yükle ve RGCN-uyumlu formata dönüştür."""
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    
    data = torch.load(path, weights_only=False)
    
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    
    edge_index_list = []
    edge_type_list = []
    
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    
    if not edge_index_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(edge_index_list, dim=1)
        edge_type = torch.cat(edge_type_list)
    
    return {
        'x': x,
        'edge_index': edge_index,
        'edge_type': edge_type,
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

# Tüm gerekli yılların snapshot'larını önbelleğe al
all_years = set(pos_df['year'].unique()) | set(neg_df['year'].unique())
snapshots = {}
for year in sorted(all_years):
    snap = load_snapshot(year)
    if snap is not None:
        snapshots[year] = snap

print(f'Yüklenen snapshot sayısı: {len(snapshots)}')

# Özellik boyutu
NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Düğüm özellik boyutu: {NUM_FEATURES}')

In [ ]:
# Eğitim örneklerini hazırla
# Her örnek: (year, members_cow_codes, label, duration, cohesion)

def parse_members(members_str):
    """members_str sütununu (virgülle ayrılmış string) list of int'e çevir."""
    if isinstance(members_str, list):
        return [int(x) for x in members_str]
    if isinstance(members_str, str):
        return [int(x) for x in members_str.split(',')]
    return []

def build_samples(df, label):
    """DataFrame'den eğitim örnekleri oluştur."""
    samples = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots:
            continue
        
        members = parse_members(row['members_str'])
        code_to_idx = snapshots[year]['code_to_idx']
        
        # Sadece snapshot'ta bulunan üyeleri al
        valid_members = [m for m in members if m in code_to_idx]
        if len(valid_members) < 2:  # en az 2 üye gerekli
            continue
        
        sample = {
            'year': year,
            'members': valid_members,
            'member_idx': [code_to_idx[m] for m in valid_members],
            'label': label,
            'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
            'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
        }
        samples.append(sample)
    return samples

pos_samples = build_samples(pos_df, label=1)
neg_samples = build_samples(neg_df, label=0)
all_samples = pos_samples + neg_samples

# Zamansal bölme
train_samples = [s for s in all_samples if s['year'] <= TRAIN_END]
val_samples = [s for s in all_samples if TRAIN_END < s['year'] <= VAL_END]
test_samples = [s for s in all_samples if s['year'] > VAL_END]

print(f'Toplam örnek: {len(all_samples)} (pos: {len(pos_samples)}, neg: {len(neg_samples)})')
print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')
print(f'\nTrain pos/neg: {sum(1 for s in train_samples if s["label"]==1)}/{sum(1 for s in train_samples if s["label"]==0)}')
print(f'Val pos/neg:   {sum(1 for s in val_samples if s["label"]==1)}/{sum(1 for s in val_samples if s["label"]==0)}')
print(f'Test pos/neg:  {sum(1 for s in test_samples if s["label"]==1)}/{sum(1 for s in test_samples if s["label"]==0)}')

## 2. Model Tanımı

In [ ]:
class RGCNEncoder(nn.Module):
    """2-katmanlı RGCN encoder (04a ile aynı mimari)."""
    
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_relations, dropout=0.2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    
    def forward(self, x, edge_index, edge_type):
        h = self.conv1(x, edge_index, edge_type)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type)
        h = self.norm2(h)
        return h


class CoalitionHead(nn.Module):
    """Koalisyon skorlama başlığı: DeepSets + pairwise bilinear (vektörize)."""
    
    def __init__(self, embed_dim, hidden_dim=64, lambda_pair=0.5):
        super().__init__()
        self.lambda_pair = lambda_pair
        
        # DeepSets MLP: mean-pooled embedding → skor
        self.set_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )
        
        # Bilinear pairwise etkileşim (vektörize edilecek)
        self.bilinear = nn.Bilinear(embed_dim, embed_dim, 1)
        
        # Yardımcı başlıklar
        self.duration_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Softplus(),
        )
        
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z_members):
        """
        Args:
            z_members: (K, D) — koalisyon üyelerinin embedding'leri
        """
        K = z_members.shape[0]
        
        # 1. Mean pooling
        z_pool = z_members.mean(dim=0, keepdim=True)  # (1, D)
        set_score = self.set_mlp(z_pool).squeeze()
        
        # 2. Pairwise bilinear — vektörize (for-loop yerine)
        if K >= 2:
            # Üst üçgen indeksleri (i < j)
            idx_i, idx_j = torch.triu_indices(K, K, offset=1, device=z_members.device)
            z_i = z_members[idx_i]  # (n_pairs, D)
            z_j = z_members[idx_j]  # (n_pairs, D)
            pair_scores = self.bilinear(z_i, z_j).squeeze(-1)  # (n_pairs,)
            pair_mean = pair_scores.mean()
        else:
            pair_mean = torch.tensor(0.0, device=z_members.device)
        
        score = set_score + self.lambda_pair * pair_mean
        
        # Yardımcı tahminler
        duration_pred = self.duration_head(z_pool).squeeze()
        cohesion_pred = self.cohesion_head(z_pool).squeeze()
        
        return score, duration_pred, cohesion_pred
    
    def compute_value(self, z_members):
        """Sadece v(S) skorunu hesapla (Shapley hesabı için)."""
        with torch.no_grad():
            score, _, _ = self.forward(z_members)
            return torch.sigmoid(score).item()


class CoalitionGNN(nn.Module):
    """Tam model: RGCN encoder + coalition scoring head."""
    
    def __init__(self, num_features, hidden_dim, embed_dim,
                 num_relations, head_hidden=64, dropout=0.2):
        super().__init__()
        self.encoder = RGCNEncoder(
            in_channels=num_features,
            hidden_channels=hidden_dim,
            out_channels=embed_dim,
            num_relations=num_relations,
            dropout=dropout,
        )
        self.head = CoalitionHead(embed_dim, hidden_dim=head_hidden)
    
    def encode_graph(self, x, edge_index, edge_type):
        """Tüm grafı encode et, düğüm embedding'leri döner."""
        return self.encoder(x, edge_index, edge_type)
    
    def score_coalition(self, z_all, member_idx):
        """Belirli üyelerin embedding'lerini alıp koalisyon skorla."""
        z_members = z_all[member_idx]  # (K, D)
        return self.head(z_members)


# Model oluştur
HIDDEN_DIM = 128
EMBED_DIM = 128
HEAD_HIDDEN = 64
DROPOUT = 0.2

model = CoalitionGNN(
    num_features=NUM_FEATURES,
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    num_relations=NUM_RELATIONS,
    head_hidden=HEAD_HIDDEN,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parametreleri: {n_params:,}')
print(f'\nModel yapısı:\n{model}')

In [ ]:
# Pretrained encoder ağırlıklarını yükle
pretrain_path = os.path.join(MODEL_DIR, 'encoder_pretrained.pt')

if os.path.exists(pretrain_path):
    checkpoint = torch.load(pretrain_path, weights_only=False)
    model.encoder.load_state_dict(checkpoint['encoder_state_dict'])
    print(f'✅ Pretrained encoder yüklendi')
    print(f'   Pretrain epoch: {checkpoint["pretrain_epoch"]}, val_loss: {checkpoint["pretrain_val_loss"]:.4f}')
else:
    print('⚠️  Pretrained encoder bulunamadı — random initialization ile devam ediliyor')
    print(f'   Beklenen yol: {pretrain_path}')

## 3. Eğitim Döngüsü

**Faz 1** (epoch 1-50): Encoder frozen, sadece coalition head eğitilir.  
**Faz 2** (epoch 51+): End-to-end, encoder LR = head LR / 10.

In [ ]:
# Eğitim hiperparametreleri
PHASE1_EPOCHS = 10   # Head ısınması (frozen encoder) — 10 epoch yeterli
PHASE2_EPOCHS = 190  # End-to-end fine-tuning
TOTAL_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS  # 200 toplam

HEAD_LR = 1e-3
ENCODER_LR = 1e-4  # head LR / 10
WEIGHT_DECAY = 1e-5
PATIENCE = 30

# Class weight: neg/pos oranı ≈ 2.5 → pozitifi kaçırmak 2.5x daha maliyetli
POS_WEIGHT = 2.5

# Loss ağırlıkları
LAMBDA_MAIN = 1.0       # contrastive (BCE)
LAMBDA_DURATION = 0.3   # duration regression
LAMBDA_COHESION = 0.3   # cohesion regression

# Faz 1: Encoder'ı dondur
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)

print(f'Faz 1: Encoder frozen ({PHASE1_EPOCHS} epoch)')
print(f'Faz 2: End-to-end ({PHASE2_EPOCHS} epoch, patience={PATIENCE})')
print(f'POS_WEIGHT: {POS_WEIGHT}')
print(f'Early stopping: VAL F1 (maximize)')
print(f'  Eğitilebilir parametre (faz 1): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
def encode_all_years(model, snapshots, device, no_grad=False):
    """Tüm yılların embedding'lerini bir seferde hesapla (cache)."""
    embeddings = {}
    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for year, snap in snapshots.items():
            x = snap['x'].to(device)
            ei = snap['edge_index'].to(device)
            et = snap['edge_type'].to(device)
            z = model.encode_graph(x, ei, et)
            embeddings[year] = z
    return embeddings


def compute_batch_loss_cached(model, samples, embeddings, device):
    """Önceden hesaplanmış embedding'lerle batch loss hesapla (hızlı)."""
    all_scores = []
    all_labels = []
    all_dur_pred = []
    all_dur_true = []
    all_coh_pred = []
    all_coh_true = []
    
    for s in samples:
        z_all = embeddings[s['year']]
        member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
        score, dur_pred, coh_pred = model.head(z_all[member_idx])
        
        all_scores.append(score)
        all_labels.append(s['label'])
        
        if s['label'] == 1:
            all_dur_pred.append(dur_pred)
            all_dur_true.append(s['duration'])
            all_coh_pred.append(coh_pred)
            all_coh_true.append(s['cohesion'])
    
    # Ana loss: BCE with class weight
    scores_t = torch.stack(all_scores)
    labels_t = torch.tensor(all_labels, dtype=torch.float, device=device)
    pos_weight = torch.tensor([POS_WEIGHT], device=device)
    main_loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_weight)
    
    # F1 hesabı
    with torch.no_grad():
        preds = (scores_t > 0).float()
        accuracy = (preds == labels_t).float().mean().item()
        tp = ((preds == 1) & (labels_t == 1)).sum().float()
        fp = ((preds == 1) & (labels_t == 0)).sum().float()
        fn = ((preds == 0) & (labels_t == 1)).sum().float()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        f1_val = f1.item()
    
    # Yardımcı loss: duration
    if all_dur_pred:
        dur_pred_t = torch.stack(all_dur_pred)
        dur_true_t = torch.tensor(all_dur_true, dtype=torch.float, device=device)
        dur_loss = F.mse_loss(dur_pred_t, torch.log1p(dur_true_t))
    else:
        dur_loss = torch.tensor(0.0, device=device)
    
    # Yardımcı loss: cohesion
    if all_coh_pred:
        coh_pred_t = torch.stack(all_coh_pred)
        coh_true_t = torch.tensor(all_coh_true, dtype=torch.float, device=device)
        valid_coh = coh_true_t > 0
        if valid_coh.sum() > 0:
            coh_loss = F.mse_loss(coh_pred_t[valid_coh], coh_true_t[valid_coh])
        else:
            coh_loss = torch.tensor(0.0, device=device)
    else:
        coh_loss = torch.tensor(0.0, device=device)
    
    total_loss = (
        LAMBDA_MAIN * main_loss +
        LAMBDA_DURATION * dur_loss +
        LAMBDA_COHESION * coh_loss
    )
    
    return total_loss, main_loss.item(), dur_loss.item(), coh_loss.item(), accuracy, f1_val

In [ ]:
# Mini-batch oluşturucu
BATCH_SIZE = 64

def make_batches(samples, batch_size=BATCH_SIZE, shuffle=True):
    """Örnekleri mini-batch'lere böl."""
    indices = list(range(len(samples)))
    if shuffle:
        random.shuffle(indices)
    
    batches = []
    for i in range(0, len(indices), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch = [samples[j] for j in batch_idx]
        batches.append(batch)
    return batches

In [ ]:
# Eğitim döngüsü — epoch-level F1 ile doğru early stopping
best_val_f1 = 0.0
patience_counter = 0
history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_prec': [], 'val_rec': [], 'val_auc': [],
}

def run_epoch_eval(model, samples, device, compute_auc=False):
    """Tüm örnekler üzerinde tek seferde metrik hesapla (doğru F1)."""
    model.eval()
    all_scores = []
    all_labels = []
    
    with torch.no_grad():
        embeddings = encode_all_years(model, snapshots, device, no_grad=True)
        for s in samples:
            z_all = embeddings[s['year']]
            member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
            score, _, _ = model.head(z_all[member_idx])
            all_scores.append(score.item())
            all_labels.append(s['label'])
    
    scores_arr = np.array(all_scores)
    labels_arr = np.array(all_labels)
    probs_arr = 1 / (1 + np.exp(-scores_arr))  # sigmoid
    preds_arr = (scores_arr > 0).astype(int)
    
    # Metrikler
    tp = ((preds_arr == 1) & (labels_arr == 1)).sum()
    fp = ((preds_arr == 1) & (labels_arr == 0)).sum()
    fn = ((preds_arr == 0) & (labels_arr == 1)).sum()
    tn = ((preds_arr == 0) & (labels_arr == 0)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy = (preds_arr == labels_arr).mean()
    
    # ROC-AUC
    from sklearn.metrics import roc_auc_score
    try:
        auc = roc_auc_score(labels_arr, probs_arr)
    except:
        auc = 0.0
    
    # Loss
    scores_t = torch.tensor(scores_arr, dtype=torch.float, device=device)
    labels_t = torch.tensor(labels_arr, dtype=torch.float, device=device)
    pos_weight = torch.tensor([POS_WEIGHT], device=device)
    loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_weight).item()
    
    return {
        'loss': loss, 'accuracy': accuracy, 'f1': f1,
        'precision': precision, 'recall': recall, 'auc': auc,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    }


def compute_batch_loss_online(model, samples, device):
    """Batch-bazlı encode + score (Faz 2 için, grad akışı korunur)."""
    by_year = {}
    for s in samples:
        by_year.setdefault(s['year'], []).append(s)
    
    all_scores = []
    all_labels = []
    all_dur_pred = []
    all_dur_true = []
    all_coh_pred = []
    all_coh_true = []
    
    for year, year_samples in by_year.items():
        snap = snapshots[year]
        x = snap['x'].to(device)
        ei = snap['edge_index'].to(device)
        et = snap['edge_type'].to(device)
        z_all = model.encode_graph(x, ei, et)
        
        for s in year_samples:
            member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
            score, dur_pred, coh_pred = model.head(z_all[member_idx])
            all_scores.append(score)
            all_labels.append(s['label'])
            if s['label'] == 1:
                all_dur_pred.append(dur_pred)
                all_dur_true.append(s['duration'])
                all_coh_pred.append(coh_pred)
                all_coh_true.append(s['cohesion'])
    
    scores_t = torch.stack(all_scores)
    labels_t = torch.tensor(all_labels, dtype=torch.float, device=device)
    pos_weight = torch.tensor([POS_WEIGHT], device=device)
    main_loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_weight)
    
    if all_dur_pred:
        dur_pred_t = torch.stack(all_dur_pred)
        dur_true_t = torch.tensor(all_dur_true, dtype=torch.float, device=device)
        dur_loss = F.mse_loss(dur_pred_t, torch.log1p(dur_true_t))
    else:
        dur_loss = torch.tensor(0.0, device=device)
    
    if all_coh_pred:
        coh_pred_t = torch.stack(all_coh_pred)
        coh_true_t = torch.tensor(all_coh_true, dtype=torch.float, device=device)
        valid_coh = coh_true_t > 0
        if valid_coh.sum() > 0:
            coh_loss = F.mse_loss(coh_pred_t[valid_coh], coh_true_t[valid_coh])
        else:
            coh_loss = torch.tensor(0.0, device=device)
    else:
        coh_loss = torch.tensor(0.0, device=device)
    
    total_loss = LAMBDA_MAIN * main_loss + LAMBDA_DURATION * dur_loss + LAMBDA_COHESION * coh_loss
    return total_loss


print(f'Eğitim: {TOTAL_EPOCHS} epoch ({PHASE1_EPOCHS} frozen + {PHASE2_EPOCHS} end-to-end)')
print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Batch: {BATCH_SIZE}')
print(f'Early stopping: VAL F1 — epoch-level (patience={PATIENCE})')
print('='*80)

# Faz 1 cache
cached_embeddings = encode_all_years(model, snapshots, device, no_grad=True)

for epoch in range(1, TOTAL_EPOCHS + 1):
    
    # --- Faz geçişi ---
    if epoch == PHASE1_EPOCHS + 1:
        print(f'\n{"="*80}')
        print(f'FAZ 2 BAŞLIYOR: Encoder unfrozen, end-to-end fine-tuning')
        print(f'{"="*80}\n')
        
        for param in model.encoder.parameters():
            param.requires_grad = True
        
        optimizer = torch.optim.Adam([
            {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
            {'params': model.head.parameters(), 'lr': HEAD_LR},
        ], weight_decay=WEIGHT_DECAY)
        
        patience_counter = 0
        cached_embeddings = None
        
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Eğitilebilir parametre: {n_trainable:,}')
    
    # --- Train (gradient step) ---
    model.train()
    epoch_train_loss = []
    
    batches = make_batches(train_samples)
    for batch in batches:
        optimizer.zero_grad()
        
        if cached_embeddings is not None:
            loss, _, _, _, _, _ = compute_batch_loss_cached(model, batch, cached_embeddings, device)
        else:
            loss = compute_batch_loss_online(model, batch, device)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_train_loss.append(loss.item())
    
    # --- Epoch-level evaluation ---
    train_m = run_epoch_eval(model, train_samples, device)
    val_m = run_epoch_eval(model, val_samples, device)
    
    history['train_loss'].append(np.mean(epoch_train_loss))
    history['train_acc'].append(train_m['accuracy'])
    history['train_f1'].append(train_m['f1'])
    history['val_loss'].append(val_m['loss'])
    history['val_acc'].append(val_m['accuracy'])
    history['val_f1'].append(val_m['f1'])
    history['val_prec'].append(val_m['precision'])
    history['val_rec'].append(val_m['recall'])
    history['val_auc'].append(val_m['auc'])
    
    # Early stopping: VAL F1
    if val_m['f1'] > best_val_f1:
        best_val_f1 = val_m['f1']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': best_val_f1,
            'val_acc': val_m['accuracy'],
            'val_auc': val_m['auc'],
            'val_loss': val_m['loss'],
            'config': {
                'num_features': NUM_FEATURES,
                'hidden_dim': HIDDEN_DIM,
                'embed_dim': EMBED_DIM,
                'num_relations': NUM_RELATIONS,
                'head_hidden': HEAD_HIDDEN,
                'dropout': DROPOUT,
                'pos_weight': POS_WEIGHT,
            }
        }, os.path.join(MODEL_DIR, 'coalition_model_best.pt'))
        marker = ' ★'
    else:
        patience_counter += 1
        marker = ''
    
    # Loglama
    if epoch % 10 == 0 or epoch == 1 or epoch == PHASE1_EPOCHS + 1 or marker:
        print(
            f'Epoch {epoch:3d} | '
            f'Train: loss={np.mean(epoch_train_loss):.4f}, F1={train_m["f1"]:.3f}, acc={train_m["accuracy"]:.3f} | '
            f'Val: loss={val_m["loss"]:.4f}, F1={val_m["f1"]:.3f}, acc={val_m["accuracy"]:.3f}, '
            f'P={val_m["precision"]:.3f}, R={val_m["recall"]:.3f}, '
            f'AUC={val_m["auc"]:.3f} '
            f'[TP={val_m["tp"]}, FP={val_m["fp"]}, FN={val_m["fn"]}]'
            f'{marker}'
        )
    
    if patience_counter >= PATIENCE:
        print(f'\n⚠️  Early stopping: {PATIENCE} epoch boyunca F1 iyileşmedi (epoch {epoch})')
        break

print(f'\n✅ Eğitim tamamlandı.')
print(f'   En iyi val F1: {best_val_f1:.4f}')
print(f'   Model: {os.path.join(MODEL_DIR, "coalition_model_best.pt")}')

## 4. Eğitim Grafikleri

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Toplam loss
axes[0, 0].plot(history['train_loss'], label='Train', alpha=0.8)
axes[0, 0].plot(history['val_loss'], label='Val', alpha=0.8)
axes[0, 0].axvline(x=PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Faz 2 başlangıcı')
axes[0, 0].set_title('Toplam Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history['train_acc'], label='Train', alpha=0.8)
axes[0, 1].plot(history['val_acc'], label='Val', alpha=0.8)
axes[0, 1].axvline(x=PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Faz 2')
axes[0, 1].set_title('Accuracy (Koalisyon Geçerlilik)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(0.4, 1.0)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Duration loss
axes[1, 0].plot(history['train_dur'], label='Train', alpha=0.8)
axes[1, 0].plot(history['val_dur'], label='Val', alpha=0.8)
axes[1, 0].axvline(x=PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Duration Loss (MSE, log-scale)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Cohesion loss
axes[1, 1].plot(history['train_coh'], label='Train', alpha=0.8)
axes[1, 1].plot(history['val_coh'], label='Val', alpha=0.8)
axes[1, 1].axvline(x=PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Cohesion Loss (MSE)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'model_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Test Seti Değerlendirmesi

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)

# En iyi modeli yükle
checkpoint = torch.load(os.path.join(MODEL_DIR, 'coalition_model_best.pt'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'En iyi model yüklendi (epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.4f})')

# Test seti üzerinde tahmin
test_scores = []
test_labels = []
test_probs = []

with torch.no_grad():
    for s in test_samples:
        snap = snapshots[s['year']]
        x = snap['x'].to(device)
        ei = snap['edge_index'].to(device)
        et = snap['edge_type'].to(device)
        
        z_all = model.encode_graph(x, ei, et)
        member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
        score, _, _ = model.score_coalition(z_all, member_idx)
        
        test_scores.append(score.item())
        test_labels.append(s['label'])
        test_probs.append(torch.sigmoid(score).item())

test_labels = np.array(test_labels)
test_probs = np.array(test_probs)

# --- Optimal eşik bulma (F1 maksimize) ---
from sklearn.metrics import precision_recall_curve

prec_vals, rec_vals, thresholds = precision_recall_curve(test_labels, test_probs)
f1_vals = 2 * prec_vals[:-1] * rec_vals[:-1] / (prec_vals[:-1] + rec_vals[:-1] + 1e-8)
best_threshold_idx = np.argmax(f1_vals)
OPTIMAL_THRESHOLD = thresholds[best_threshold_idx]
print(f'\nOptimal eşik (F1 maksimize): {OPTIMAL_THRESHOLD:.4f}')
print(f'  Bu eşikte: F1={f1_vals[best_threshold_idx]:.4f}, '
      f'Prec={prec_vals[best_threshold_idx]:.4f}, Rec={rec_vals[best_threshold_idx]:.4f}')

# Varsayılan (0.5) ve optimal eşikle karşılaştır
print('\n' + '='*60)
print('TEST SETİ SONUÇLARI (2010-2016)'.center(60))
print('='*60)
print(f'\nÖrnek sayısı: {len(test_labels)} (pos: {test_labels.sum()}, neg: {(1-test_labels).sum():.0f})')

for threshold_name, threshold in [('Varsayılan (0.5)', 0.5), ('Optimal', OPTIMAL_THRESHOLD)]:
    test_preds = (test_probs >= threshold).astype(int)
    print(f'\n--- Eşik: {threshold_name} ({threshold:.4f}) ---')
    print(f'Accuracy:  {accuracy_score(test_labels, test_preds):.4f}')
    print(f'Precision: {precision_score(test_labels, test_preds, zero_division=0):.4f}')
    print(f'Recall:    {recall_score(test_labels, test_preds, zero_division=0):.4f}')
    print(f'F1:        {f1_score(test_labels, test_preds, zero_division=0):.4f}')

# Eşik-bağımsız metrikler
print(f'\n--- Eşik-bağımsız metrikler ---')
print(f'ROC-AUC:   {roc_auc_score(test_labels, test_probs):.4f}')
print(f'PR-AUC:    {average_precision_score(test_labels, test_probs):.4f}')

# Optimal eşikle detaylı rapor
test_preds_optimal = (test_probs >= OPTIMAL_THRESHOLD).astype(int)
print(f'\nSınıflandırma Raporu (optimal eşik={OPTIMAL_THRESHOLD:.3f}):')
print(classification_report(test_labels, test_preds_optimal, target_names=['Geçersiz', 'Geçerli']))

print(f'Confusion Matrix:')
cm = confusion_matrix(test_labels, test_preds_optimal)
print(f'  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}')
print(f'  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}')

In [ ]:
# ROC ve PR eğrileri
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
fpr, tpr, _ = roc_curve(test_labels, test_probs)
auc_score = roc_auc_score(test_labels, test_probs)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC={auc_score:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Eğrisi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall
prec, rec, _ = precision_recall_curve(test_labels, test_probs)
ap_score = average_precision_score(test_labels, test_probs)
axes[1].plot(rec, prec, 'r-', linewidth=2, label=f'PR (AP={ap_score:.3f})')
baseline = test_labels.mean()
axes[1].axhline(y=baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Eğrisi')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'model_test_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Shapley Değerleri — Üye Marjinal Katkıları

Monte Carlo yaklaşımıyla her üyenin koalisyondaki marjinal değerini hesaplıyoruz.

**Shapley formülü:**
$$\phi_i(v) = \frac{1}{M} \sum_{m=1}^{M} [v(S^m_{\pi} \cup \{i\}) - v(S^m_{\pi})]$$

Burada $S^m_{\pi}$, rastgele permütasyon $\pi$'de $i$'den önceki oyuncular kümesi.

In [ ]:
def compute_coalition_value(model, z_all, member_idx, device, use_logit=False):
    """Koalisyon değeri v(S).
    
    use_logit=False → sigmoid olasılık [0,1] (Shapley için)
    use_logit=True  → raw logit (stabilite analizi için, satürasyonu önler)
    """
    if len(member_idx) < 2:
        return 0.0
    idx_tensor = torch.tensor(member_idx, dtype=torch.long, device=device)
    z_members = z_all[idx_tensor]
    
    if use_logit:
        # Raw logit — sigmoid satürasyonu olmadan gerçek değer farkları korunur
        score, _, _ = model.head(z_members)
        return score.item()
    else:
        return model.head.compute_value(z_members)


def monte_carlo_shapley(model, z_all, member_idx, device, n_samples=200, use_logit=False):
    """Monte Carlo Shapley değerleri."""
    K = len(member_idx)
    shapley = {idx: 0.0 for idx in member_idx}
    
    for _ in range(n_samples):
        perm = list(member_idx)
        random.shuffle(perm)
        
        prev_value = 0.0
        coalition_so_far = []
        
        for player in perm:
            coalition_so_far.append(player)
            if len(coalition_so_far) >= 2:
                curr_value = compute_coalition_value(model, z_all, coalition_so_far, device, use_logit=use_logit)
            else:
                curr_value = 0.0
            
            marginal = curr_value - prev_value
            shapley[player] += marginal / n_samples
            prev_value = curr_value
    
    return shapley


print('✅ Shapley fonksiyonları tanımlandı (200 MC permütasyon/koalisyon)')
print('   use_logit=False → sigmoid (Shapley üye katkıları)')
print('   use_logit=True  → raw logit (stabilite analizi, satürasyon önlenir)')

In [ ]:
# COW kodu → ülke ismi eşlemesi
try:
    country_master = pd.read_parquet(os.path.join(CANON_DIR, 'nodes.parquet'))
    # year bazlı ilk kayıttan ülke ismi al
    cow_to_name = dict(zip(country_master['cow_code'], country_master['state_name']))
except:
    cow_to_name = {}
    print('⚠️ nodes.parquet yüklenemedi, COW kodları kullanılacak')

# Pozitif örneklerden ilginç koalisyonları seç (3-12 üye arası, farklı yıllardan)
analysis_samples = []
seen_sizes = set()
for s in pos_samples:
    K = len(s['member_idx'])
    if K < 3 or K > 12:
        continue
    if s['year'] not in snapshots:
        continue
    # Farklı boyutlar ve yıllardan örnekle
    size_decade = (K, s['year'] // 10)
    if size_decade not in seen_sizes:
        seen_sizes.add(size_decade)
        analysis_samples.append(s)
    if len(analysis_samples) >= 20:
        break

print(f'Shapley analizi için seçilen koalisyon sayısı: {len(analysis_samples)}')
print(f'Üye aralığı: {min(len(s["member_idx"]) for s in analysis_samples)}-{max(len(s["member_idx"]) for s in analysis_samples)}')
print(f'Yıl aralığı: {min(s["year"] for s in analysis_samples)}-{max(s["year"] for s in analysis_samples)}')

# Shapley hesabı
shapley_results = []

model.eval()
with torch.no_grad():
    for i, sample in enumerate(tqdm(analysis_samples, desc='Shapley hesabı')):
        year = sample['year']
        snap = snapshots[year]
        x = snap['x'].to(device)
        ei = snap['edge_index'].to(device)
        et = snap['edge_type'].to(device)
        
        z_all = model.encode_graph(x, ei, et)
        member_idx = sample['member_idx']
        
        # v(N) — büyük koalisyonun değeri
        v_grand = compute_coalition_value(model, z_all, member_idx, device)
        
        # Shapley
        shapley = monte_carlo_shapley(model, z_all, member_idx, device, n_samples=200)
        
        # COW koduna dönüştür
        idx_to_cow = {v: k for k, v in snap['code_to_idx'].items()}
        shapley_named = {}
        for idx, val in shapley.items():
            cow = idx_to_cow[idx]
            name = cow_to_name.get(cow, f'COW_{cow}')
            shapley_named[name] = val
        
        shapley_results.append({
            'year': year,
            'members': [cow_to_name.get(idx_to_cow[idx], f'COW_{idx_to_cow[idx]}') for idx in member_idx],
            'v_grand': v_grand,
            'shapley': shapley_named,
            'member_idx': member_idx,
        })

# Sonuçları yazdır (paylaşılabilir format)
print(f'\n{"="*70}')
print(f'{"SHAPLEY DEĞERLERİ — KOALISYON ÜYE KATKILARI":^70}')
print(f'{"="*70}')

for i, res in enumerate(shapley_results):
    sorted_sv = sorted(res['shapley'].items(), key=lambda x: x[1], reverse=True)
    print(f'\n{"─"*70}')
    print(f'Koalisyon #{i+1} | Yıl: {res["year"]} | Üye: {len(res["members"])} | v(N)={res["v_grand"]:.4f}')
    print(f'Üyeler: {", ".join(res["members"])}')
    print(f'{"─"*70}')
    print(f'  {"Ülke":<25} {"Shapley φᵢ":>12} {"Katkı %":>10}')
    total_sv = sum(abs(v) for _, v in sorted_sv)
    for name, val in sorted_sv:
        pct = (val / total_sv * 100) if total_sv > 0 else 0
        bar = '█' * int(abs(pct) / 5) if pct > 0 else '░' * int(abs(pct) / 5)
        sign = '+' if val >= 0 else '-'
        print(f'  {name:<25} {sign}{abs(val):>10.4f} {pct:>+8.1f}% {bar}')

print(f'\n{"="*70}')
print(f'Toplam {len(shapley_results)} koalisyon analiz edildi.')
print(f'φᵢ > 0: üye koalisyona değer katıyor')
print(f'φᵢ < 0: üye koalisyonu zayıflatıyor (potansiyel ayrılma adayı)')

In [ ]:
# Shapley görselleştirmesi — en yüksek v(N) olan koalisyonlar
import matplotlib.pyplot as plt

# En yüksek v(N) olan 6 koalisyonu seç
top_coalitions = sorted(shapley_results, key=lambda x: x['v_grand'], reverse=True)[:6]

n_show = len(top_coalitions)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, res in enumerate(top_coalitions):
    ax = axes[i]
    sorted_sv = sorted(res['shapley'].items(), key=lambda x: x[1], reverse=True)
    names = [n[:15] for n, _ in sorted_sv]  # kısa isim
    values = [v for _, v in sorted_sv]
    
    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in values]
    ax.barh(range(len(names)), values, color=colors, alpha=0.8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Shapley φᵢ')
    ax.set_title(f'{res["year"]} | {len(res["members"])} üye | v(N)={res["v_grand"]:.3f}', fontsize=9)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')

for j in range(n_show, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Koalisyon Üyelerinin Shapley Değerleri (Marjinal Katkı)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'shapley_values.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Koalisyon Stabilite Analizi

**Çekirdek (Core) kontrolü:** Bir koalisyon "çekirdekte" ise hiçbir alt-koalisyon ayrılarak daha iyi yapamaz.

Basitleştirilmiş test: Her alt-küme S ⊂ N için v(S) ≤ Σᵢ∈S φᵢ olmalı.

In [ ]:
from itertools import combinations

def find_blocking_coalitions(model, z_all, member_idx, device, max_subsets=200):
    """Bloklayan koalisyonları bul: v(S) > v(N) olan alt-kümeler.
    
    Süperadditif olmayan oyunlarda klasik 'core' anlamsız.
    Bunun yerine: hangi alt-gruplar büyük koalisyondan daha güçlü?
    Bu, tarihsel olarak 'doğal' alt-ittifakları tespit eder.
    """
    K = len(member_idx)
    v_grand = compute_coalition_value(model, z_all, member_idx, device, use_logit=True)
    
    # Tüm olası alt-kümeleri oluştur
    all_subsets = []
    for size in range(2, K):
        for subset in combinations(range(K), size):
            all_subsets.append(subset)
    
    if len(all_subsets) > max_subsets:
        all_subsets = random.sample(all_subsets, max_subsets)
    
    blocking = []
    weaker = []
    
    for subset_indices in all_subsets:
        subset_members = [member_idx[i] for i in subset_indices]
        v_subset = compute_coalition_value(model, z_all, subset_members, device, use_logit=True)
        
        ratio = v_subset / (v_grand + 1e-8)  # v(S)/v(N) oranı
        
        entry = {
            'subset_idx': subset_indices,
            'size': len(subset_indices),
            'v_subset': v_subset,
            'ratio': ratio,  # >1 = alt-küme daha güçlü
        }
        
        if v_subset > v_grand:
            blocking.append(entry)
        else:
            weaker.append(entry)
    
    # En güçlü bloklayan koalisyonları sırala
    blocking.sort(key=lambda x: x['v_subset'], reverse=True)
    
    return v_grand, blocking, weaker


# Analiz
print(f'{"="*70}')
print(f'{"BLOKLAYAN KOALİSYON ANALİZİ (NON-SUPERADDITIVE GAMES)":^70}')
print(f'{"="*70}')
print(f'\nSoru: Hangi alt-gruplar büyük koalisyondan daha güçlü?')
print(f'v(S) > v(N) ise S bir "blocking coalition" — büyük koalisyonu')
print(f'bozmak için yeterli teşvike sahip doğal alt-ittifak.\n')

blocking_analysis = []

with torch.no_grad():
    for i, res in enumerate(tqdm(shapley_results, desc='Blocking analizi')):
        year = res['year']
        snap = snapshots[year]
        x = snap['x'].to(device)
        ei = snap['edge_index'].to(device)
        et = snap['edge_type'].to(device)
        
        z_all = model.encode_graph(x, ei, et)
        member_idx = res['member_idx']
        
        v_grand, blocking, weaker = find_blocking_coalitions(
            model, z_all, member_idx, device
        )
        
        # En güçlü alt-küme
        best_sub = blocking[0] if blocking else None
        
        blocking_analysis.append({
            'year': year,
            'members': res['members'],
            'K': len(member_idx),
            'v_grand': v_grand,
            'n_blocking': len(blocking),
            'n_total_subsets': len(blocking) + len(weaker),
            'pct_blocking': 100 * len(blocking) / (len(blocking) + len(weaker) + 1e-8),
            'best_blocking': best_sub,
            'member_idx': member_idx,
        })

# Özet tablo
print(f'\n{"─"*70}')
print(f'  {"#":<3} {"Yıl":<6} {"K":<3} {"v(N)":<8} {"Block%":<8} {"En güçlü alt-küme":30}')
print(f'{"─"*70}')

for i, r in enumerate(blocking_analysis):
    if r['best_blocking']:
        best_names = [r['members'][j] for j in r['best_blocking']['subset_idx']]
        best_str = f'{",".join(best_names[:4])}{"..." if len(best_names)>4 else ""} ({r["best_blocking"]["size"]}üye, v={r["best_blocking"]["v_subset"]:.1f})'
    else:
        best_str = '—'
    
    # Bloklama yüzdesi: ne kadar çok alt-küme büyük koalisyondan güçlüyse o kadar "kırılgan"
    fragility = '🔴' if r['pct_blocking'] > 60 else '��' if r['pct_blocking'] > 30 else '🟢'
    
    print(f'  {i+1:<3} {r["year"]:<6} {r["K"]:<3} {r["v_grand"]:<8.2f} {r["pct_blocking"]:<6.1f}%{fragility} {best_str}')

print(f'{"─"*70}')

# Kırılganlık özeti
high_frag = [r for r in blocking_analysis if r['pct_blocking'] > 60]
mid_frag = [r for r in blocking_analysis if 30 < r['pct_blocking'] <= 60]
low_frag = [r for r in blocking_analysis if r['pct_blocking'] <= 30]

print(f'\n📊 KIRILGANLIK ÖZETİ:')
print(f'   🟢 Sağlam (≤30% blocking):  {len(low_frag)}/20')
print(f'   🟡 Orta (30-60% blocking):   {len(mid_frag)}/20')
print(f'   🔴 Kırılgan (>60% blocking): {len(high_frag)}/20')

# Detaylı analiz: en ilginç vakalar
print(f'\n{"─"*70}')
print(f'DETAYLI ANALİZ — SEÇİLMİŞ VAKALAR:')
print(f'{"─"*70}')

for r in blocking_analysis:
    if r['best_blocking'] is None:
        continue
    
    best = r['best_blocking']
    best_names = [r['members'][j] for j in best['subset_idx']]
    excluded = [m for j, m in enumerate(r['members']) if j not in best['subset_idx']]
    
    print(f'\n  {r["year"]} | {r["K"]} üyeli koalisyon | v(N)={r["v_grand"]:.2f}')
    print(f'  Tam kadro: {", ".join(r["members"])}')
    print(f'  En güçlü çekirdek ({best["size"]} üye): {", ".join(best_names)} → v(S)={best["v_subset"]:.2f}')
    print(f'  Dışarıda kalanlar: {", ".join(excluded)}')
    print(f'  Yorum: Dışarıda kalanlar koalisyonu zayıflatıyor (v(S) > v(N))')
    
    # Sadece ilk 8 koalisyonu detaylı göster
    if blocking_analysis.index(r) >= 7:
        print(f'\n  ... (toplam {len(blocking_analysis)} koalisyon analiz edildi)')
        break

print(f'\n{"="*70}')
print(f'GENEL YORUM:')
print(f'  Bu analiz "superadditive olmayan" bir koalisyon oyununu yansıtıyor.')
print(f'  Uluslararası ittifaklarda üye eklemek her zaman güçlendirmez —')
print(f'  ideolojik/stratejik uyumsuz üyeler koalisyonu zayıflatabilir.')
print(f'  Blocking %\'si yüksek olan koalisyonlar tarihsel olarak daha kırılgan.')
print(f'  (Varşova Paktı 1990: Arnavutluk zaten 1968\'de ayrılmıştı!)')

## 8. Sonuçları Kaydet

In [ ]:
import json

# Shapley sonuçlarını JSON olarak kaydet
shapley_export = []
for (year, members), shapley_cow in shapley_results.items():
    shapley_export.append({
        'year': int(year),
        'members': [int(m) for m in members],
        'shapley_values': {str(k): float(v) for k, v in shapley_cow.items()},
    })

with open(os.path.join(PROJECT_DIR, 'shapley_results.json'), 'w') as f:
    json.dump(shapley_export, f, indent=2)

# Stabilite sonuçlarını kaydet
stability_export = []
for r in stability_results:
    stability_export.append({
        'year': int(r['year']),
        'members': [int(m) for m in r['members']],
        'v_grand': float(r['v_grand']),
        'is_stable': bool(r['is_stable']),
        'n_violations': int(r['n_violations']),
    })

with open(os.path.join(PROJECT_DIR, 'stability_results.json'), 'w') as f:
    json.dump(stability_export, f, indent=2)

# Test metrikleri
test_metrics = {
    'accuracy': float(accuracy_score(test_labels, test_preds)),
    'precision': float(precision_score(test_labels, test_preds, zero_division=0)),
    'recall': float(recall_score(test_labels, test_preds, zero_division=0)),
    'f1': float(f1_score(test_labels, test_preds, zero_division=0)),
    'roc_auc': float(roc_auc_score(test_labels, test_probs)),
    'pr_auc': float(average_precision_score(test_labels, test_probs)),
    'n_test': len(test_labels),
    'n_pos': int(test_labels.sum()),
    'n_neg': int((1 - test_labels).sum()),
}

with open(os.path.join(PROJECT_DIR, 'test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)

print('✅ Sonuçlar kaydedildi:')
print(f'   shapley_results.json ({len(shapley_export)} koalisyon)')
print(f'   stability_results.json ({len(stability_export)} koalisyon)')
print(f'   test_metrics.json')
print(f'   coalition_model_best.pt')

## Özet

### Model Çıktıları
1. `models/coalition_model_best.pt` — eğitilmiş tam model
2. `test_metrics.json` — test seti performans metrikleri
3. `shapley_results.json` — üye marjinal katkıları
4. `stability_results.json` — çekirdek stabilite analizi
5. Görselleştirmeler: eğitim eğrileri, ROC/PR, Shapley bar grafikler

### Mimari Özeti
- **Encoder:** 2-katmanlı RGCN (128h, 128d, 4 ilişki tipi)
- **Head:** DeepSets (mean pool) + pairwise bilinear + MLP
- **Multi-task:** BCE (geçerlilik) + MSE (süre) + MSE (uyum)
- **Fine-tuning:** Aşamalı (50 epoch frozen + end-to-end)
- **Game theory:** Monte Carlo Shapley + core stability check

### Sonraki Adımlar
- Makale yazımı: bulguları raporla
- Ek analiz: koalisyon genişleme/daralma tahminleri
- Robustness: farklı negatif örnekleme stratejileri ile karşılaştırma
- Ablation: pretrain etkisi, pairwise terim katkısı